testing HDF5 conversion to other formats

In [3]:
# ensure the kernel is working
from sys import version
print(version)

3.13.1 (main, Oct 27 2025, 14:41:37) [GCC 14.2.0]


In [4]:
# import pandas as pd
import h5py
import os
from fod_config import *
from dotenv import load_dotenv
from fod3 import read_narr_lat_lon,validate_latlon,read_one_year,read_narr_timeseries, path_to_narrfile

In [5]:
load_dotenv()

True

In [6]:
os.getenv('NARR_INPUT_DIR')

'/mnt/research/ICER-RSE/clients/enviroweather/mioffset/h5'

In [7]:
narr_input_dir = h5folder = os.getenv('NARR_INPUT_DIR')
 
h5filename = path_to_narrfile(2001, narr_input_dir)
os.path.exists(h5filename)



True

In [8]:
with h5py.File(h5filename,'r') as hf:    
    print('List of arrays in this file: \n', list(hf.keys()))

List of arrays in this file: 
 ['PC', 'WD', 'WS']


**test read of hdf5 files**

In [9]:
yr = 2008
idy = 1
idx = 1
pc_1year, ws_1year, wd_1year = read_one_year(yr=yr, idy=idy, idx=idx, narr_input_dir=h5folder)
ws_1year

array([6.277, 5.044, 5.461, ..., 5.236, 5.085, 5.546], shape=(2920,))

In [10]:
def read_one_year_grid(yr:str,narr_input_dir:str):
    """read one year hf5 file, extra 3 datasets
    Files must be named like narr_PSD_1980_BC.h5
    
    Args:
        yr (int): year of data to read, embedded in filename
        idy (int): index of grid y coordinate (North/South)
        idx (int): index of grid x coordinate (East/West)
        narr_input_dir (str): path to NARR input files
    Returns:
        tuple of np arrays: timeseries values for PC, WD and WS from one grid point, all hours
    """
    
    h5f_annual_filename = path_to_narrfile(yr, narr_input_dir)
    h5f = h5py.File(h5f_annual_filename, 'r')
    # extract all values for one year
    # previously filtered at read time, like
    #  pc_1year = h5f['pc'][idy,idx,ts:te]
    pc_1year = h5f['PC']
    ws_1year = h5f['WS']
    wd_1year = h5f['WD']
    return pc_1year, ws_1year, wd_1year

In [11]:
# test grid read
pc_1year, ws_1year, wd_1year = read_one_year_grid(yr=yr, narr_input_dir=narr_input_dir)
pc_1year.shape


(277, 349, 2920)

In [12]:
# size of point data for a year
pc_1year.shape[0] * pc_1year.shape[1]

96673

size of ALL data

In [13]:
start_year = 1979; end_year = 2009
years = range(start_year, end_year+1)  # range 'stop' is the value after the top value becuse PYTHON
count_all_vals = (years[-1] - years[0]) * (pc_1year.shape[0] * pc_1year.shape[1]) * pc_1year.shape[2]

print( "number of vals in 30 yr  stack" , count_all_vals )
print( "gb" , (count_all_vals * 4)/ (1024**3) )

number of vals in 30 yr  stack 8468554800
gb 31.547825038433075


if we have the memory it is less IO to read everythign into memory but need 

In [14]:
print( "number of vals in 30 yr point stack" ,  (years[-1] - years[0]) * (pc_1year.shape[0] * pc_1year.shape[1]))

number of vals in 30 yr point stack 2900190


Psuedo code to export this to JSON. 

Note that the fod main model currently assumes there are 2920 values per year and just smushes them all together in a long time series with no time associated with them
in the JSON file should have years for each, and a process for doing the smushing

1. read in all years in 3 giant 3d matrices.   or naively 3 dictionaries with year

PC = {1979: 3d np.array, 1980: 3dnp.array. etc}
WD = etc

matrix of x,ys   or list of x,y combinations from one of the years

2. grid_points 
shape is (277, 349, 2920)
so grid points are x: (0:276) y = rang(0:348) pc_1year.shape[0] * pc_1year.shape[1] -> 96673 points

if we save by grid point that is 96K files.  

each file will be 

3. loop for each grid point x,y
point_vals = {}
for each year
    point_vals[year] = get values for x,y

4. store in AWS: 
best practices https://reintech.io/blog/organizing-data-amazon-s3-best-practices

single bucket, but unique indices depending on how data is read

object name has elements Variable (PC, WD, WS) x, y
based on above better to use x,y first and var name last

`{x}/{y}/PC.json`   <- all years in this file for this coordinate

- program needs to find only 3 object, PC, WD, WS
- each object has 2920 timepoint * 30 yrs = 

single bucket means single point of permisions setting since all data will be
read by same program

```python
# setup aws clie config and IAM user creds in folder
import boto3
# global is easier for this one-off
client = boto3.client('s3')
bucket_name  = "narr_wind_timeseries"


def saveS3(winddata, varname,x,y):
    object_name = f"{x}/{y}/{varname}_{x}_{y}.json"
    d = json.dumps(winddata)
    try:
        res = client.put_object(Body=d, Bucket=bucket_name, Key=object_name)
    except exception as e:
        print(f"can't write {object_name} to bucket")
        raise
    print(res)
```

In [15]:
# reload .env file to accommodate changes during development
load_dotenv(override = True)
print(os.getenv("REGION_NAME"))
print(os.getenv("NARR_INPUT_DIR"))
os.path.exists(os.getenv("NARR_INPUT_DIR"))

us-east-1
/mnt/research/ICER-RSE/clients/enviroweather/mioffset/h5


True

create an AWS session, but ensure to use the AWS config that's in the .env fle or set by the program instead of the default config that may be set in your `~/.aws/config` file

In [16]:
aws_config = { 
    "aws_access_key_id" : os.getenv("AWS_ACCESS_KEY_ID"), 
    "aws_secret_access_key" : os.getenv("AWS_SECRET_ACCESS_KEY"), 
    "region_name" : os.getenv("REGION_NAME") 
}

import boto3
session = boto3.Session(**aws_config)
session

Session(region_name='us-east-1')

Make an s3 client from session

In [17]:
s3_client = session.client('s3')
s3_client

In [18]:
response = s3_client.list_buckets()
print([bucket['Name'] for bucket in response['Buckets']])

bucket_list = [ bucket['Name'] for bucket in list(response['Buckets'])]
narr_bucket = os.getenv('BUCKET_NAME')
print(narr_bucket in bucket_list)

['elasticbeanstalk-us-east-1-941061635635', 'mioffsetnarr']
True


just for sharing purposes, put all the NARR files into that bucket as they are

In [19]:
narr_input_dir = os.getenv('NARR_INPUT_DIR')

yr = 1980
narr_file = path_to_narrfile(yr,narr_input_dir)
print(narr_file)
print(os.path.exists(narr_file))


/mnt/research/ICER-RSE/clients/enviroweather/mioffset/h5/narr_PSD_1980_BC.h5
True


In [20]:
# copy up all the HDF5 files in current form to S3 
# this is done
# from pathlib import Path

# for file in Path(narr_input_dir).rglob('*.h5'):
#     print(f"uploading {file.name}")
#     s3_client.upload_file(file, narr_bucket, f"narr/{file.name}")

In [ ]:
# read in all years, full grid into dictionaries keyed on year
PC = {}
WD = {}
WS = {}

for yr in range(1979,2009):
    print(yr)
    PC[yr], WD[yr], WS[yr] = read_one_year_grid(yr, narr_input_dir)


In [22]:
# get config variables - I really need to put this into a module and config class
time_flag = os.getenv("TIME_FLAG")
output_offset_dir=os.getenv("OUTPUT_OFFSET_DIR")
narr_file=os.getenv("NARR_INPUT")
narr_input_dir=os.getenv("NARR_INPUT_DIR")

In [23]:
LON, LAT = read_narr_lat_lon(narr_file)

In [24]:
# create a list of all grid coordinate pairs in X,Y space (LON/LAT)
import numpy as np
xdx = list(range(PC[1979].shape[0]))
ydx = list(range(PC[1979].shape[1]))
coords = []

for a in xdx:
    for b in  ydx:
        coords.append( [ a, b ] )
coords

[[0, 0],
 [0, 1],
 [0, 2],
 [0, 3],
 [0, 4],
 [0, 5],
 [0, 6],
 [0, 7],
 [0, 8],
 [0, 9],
 [0, 10],
 [0, 11],
 [0, 12],
 [0, 13],
 [0, 14],
 [0, 15],
 [0, 16],
 [0, 17],
 [0, 18],
 [0, 19],
 [0, 20],
 [0, 21],
 [0, 22],
 [0, 23],
 [0, 24],
 [0, 25],
 [0, 26],
 [0, 27],
 [0, 28],
 [0, 29],
 [0, 30],
 [0, 31],
 [0, 32],
 [0, 33],
 [0, 34],
 [0, 35],
 [0, 36],
 [0, 37],
 [0, 38],
 [0, 39],
 [0, 40],
 [0, 41],
 [0, 42],
 [0, 43],
 [0, 44],
 [0, 45],
 [0, 46],
 [0, 47],
 [0, 48],
 [0, 49],
 [0, 50],
 [0, 51],
 [0, 52],
 [0, 53],
 [0, 54],
 [0, 55],
 [0, 56],
 [0, 57],
 [0, 58],
 [0, 59],
 [0, 60],
 [0, 61],
 [0, 62],
 [0, 63],
 [0, 64],
 [0, 65],
 [0, 66],
 [0, 67],
 [0, 68],
 [0, 69],
 [0, 70],
 [0, 71],
 [0, 72],
 [0, 73],
 [0, 74],
 [0, 75],
 [0, 76],
 [0, 77],
 [0, 78],
 [0, 79],
 [0, 80],
 [0, 81],
 [0, 82],
 [0, 83],
 [0, 84],
 [0, 85],
 [0, 86],
 [0, 87],
 [0, 88],
 [0, 89],
 [0, 90],
 [0, 91],
 [0, 92],
 [0, 93],
 [0, 94],
 [0, 95],
 [0, 96],
 [0, 97],
 [0, 98],
 [0, 99],
 [0, 100],

In [71]:
import json

save_path = "/mnt/research/ICER-RSE/clients/enviroweather/mioffset/maaa_mioffset/tmp"
output_offset_dir=os.getenv("OUTPUT_OFFSET_DIR")
years = range(1979, 2009)

# save stacked point files... 
for coord in coords:     
    print(coord)
    x,y = coord
    pc_years = {}
    wd_years = {}
    ws_years = {}
    for yr in years:
        print(f"\t {yr}")
        wd_years[yr] = list(map(float, WD[yr][x][y]))        
        ws_years[yr] = list(map(float, WS[yr][x][y]))
        pc_years[yr] = list(map(float, PC[yr][x][y]))
        
    
    print(s3_client.put_object(
        Body=json.dumps(pc_years),
        Bucket=narr_bucket,
        Key=f"pc/pc_{x}_{y}.json"
        ))

    print(
        s3_client.put_object(
            Body=json.dumps(wd_years),
            Bucket=narr_bucket,
            Key=f"wd/wd_{x}_{y}.json"
            )
    )

    print(
        s3_client.put_object(
            Body=json.dumps(ws_years),
            Bucket=narr_bucket,
            Key=f"ws/ws_{x}_{y}.json"
        )
    )


    # with open(time_series_file, 'w') as f:
    #     f.writelines(ts_j)


[0, 0]
	 1979
	 1980
	 1981
	 1982
	 1983
	 1984
	 1985
	 1986
	 1987
	 1988
	 1989
	 1990
	 1991
	 1992
	 1993
	 1994
	 1995
	 1996
	 1997
	 1998
	 1999
	 2000
	 2001
	 2002
	 2003
	 2004
	 2005
	 2006
	 2007
	 2008
{'ResponseMetadata': {'RequestId': 'QF7K1A6NZAEBJ46X', 'HostId': 'WyyQ5hA5H7v4homYUJXumM8yGAO8LTzuYnyjWVgJmSm2RL21EvhpavbcCQ0jNkQXGMSlzK/I2VE=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'WyyQ5hA5H7v4homYUJXumM8yGAO8LTzuYnyjWVgJmSm2RL21EvhpavbcCQ0jNkQXGMSlzK/I2VE=', 'x-amz-request-id': 'QF7K1A6NZAEBJ46X', 'date': 'Sat, 11 Apr 2026 15:30:48 GMT', 'x-amz-server-side-encryption': 'AES256', 'etag': '"34c1b2729b79813960d47ea4293687a0"', 'x-amz-checksum-crc32': 'Fg+VAw==', 'x-amz-checksum-type': 'FULL_OBJECT', 'content-length': '0', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'ETag': '"34c1b2729b79813960d47ea4293687a0"', 'ChecksumCRC32': 'Fg+VAw==', 'ChecksumType': 'FULL_OBJECT', 'ServerSideEncryption': 'AES256'}
{'ResponseMetadata': {'RequestId': '0RKADY31R9EKB1AK', 

KeyboardInterrupt: 

In [66]:
# test writing and converting to np array
# ts = timeseries
import json
ts_by_year_file = "../tmp/wd_0_10.json"
with open(ts_by_year_file, "r") as f:
    ts_by_year = f.read()

ts_by_year = json.loads(ts_by_year)
ts_by_year


{'1979': [5.733,
  5.321,
  5.46,
  5.618,
  5.305,
  4.942,
  5.483,
  5.173,
  5.316,
  4.649,
  6.223,
  6.149,
  5.642,
  5.254,
  5.589,
  5.034,
  5.938,
  5.887,
  6.382,
  7.25,
  6.543,
  5.969,
  5.731,
  4.611,
  4.835,
  5.126,
  5.29,
  5.066,
  5.413,
  5.418,
  5.342,
  4.574,
  5.526,
  5.868,
  6.141,
  5.938,
  6.735,
  5.053,
  4.881,
  3.452,
  5.776,
  5.333,
  4.474,
  3.973,
  4.837,
  4.708,
  4.868,
  2.963,
  5.033,
  4.108,
  6.002,
  7.159,
  5.413,
  6.06,
  6.644,
  4.323,
  4.922,
  5.573,
  5.558,
  5.791,
  4.676,
  5.07,
  4.925,
  4.543,
  7.497,
  7.715,
  6.929,
  6.147,
  5.648,
  5.297,
  5.542,
  4.787,
  7.484,
  7.254,
  6.916,
  6.862,
  5.183,
  5.386,
  5.501,
  4.187,
  6.131,
  6.235,
  8.42,
  6.762,
  4.447,
  3.916,
  2.911,
  6.797,
  4.524,
  4.285,
  4.52,
  4.574,
  2.612,
  0.96,
  2.848,
  4.363,
  4.172,
  4.302,
  4.114,
  3.916,
  5.188,
  4.453,
  4.95,
  6.615,
  6.267,
  6.135,
  6.772,
  7.177,
  6.491,
  6.112,
  7.496,
  

In [68]:
ts_by_year_nparray = list(map(np.array, list(ts_by_year.values())))
ts_by_year_merged = np.array(np.concatenate(ts_by_year_nparray))
ts_by_year_merged

array([5.733, 5.321, 5.46 , ..., 5.162, 4.842, 5.58 ], shape=(87600,))

In [70]:
s3_client